In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from castle.utils.latent_explorer import Latent
except ImportError:
    # Fallback if package not in path
    import sys
    sys.path.append('..')
    try:
        from castle.utils.latent_explorer import Latent
    except ImportError:
        print("Please ensure you are running this notebook from the root or notebooks/ directory and castle is installed.")
        raise


In [ ]:
import numpy as np
import os

latent_path = 'temp/video_latent.npz'
if not os.path.exists(latent_path):
    print(f"File {latent_path} not found. please run Step 4 first.")
    # Dummy data for demo
    X_raw = np.random.randn(100, 768)
else:
    X_raw = np.load(latent_path)['latent']
    
print("Latent shape:", X_raw.shape)

# Initialize Latent Explorer
X = Latent(X_raw, time_window=2)  # time_window = 2 frames

In [ ]:
# Select a Cluster (or initial data as cluster 0)
target = X.select(selected_cluster='init') # or 0
# target = X.select(cluster_id=0)

In [ ]:
# UMAP Embedding
cfg_umap = {
    'n_neighbors': 30,
    'min_dist': 0.0,
    'metric': 'euclidean'
}

print("Building UMAP...")
target.build_embedding(cfg_umap)

plt.figure(figsize=(10, 8))
target.plot_embedding()
plt.title('UMAP Embedding')
plt.show()

In [ ]:
# Clustering (DBSCAN)
cfg_dbscan = {
    'eps': 2.0,
    'min_samples': 5
}

print("Running DBSCAN...")
target.build_cluster(method='dbscan', configs=cfg_dbscan)

plt.figure(figsize=(10, 8))
target.plot_embedding()
plt.title('Clustered UMAP')
plt.show()

In [ ]:
# Import clusters back to main Latent object
# You need to define labels for clusters or it uses default
# target.label_cluster(0, 'Behavior A', 'red')

# X.import_local_latent(target) 
# Note: import_local_latent expects the local latent to have export dict populated if using that logic
# Actually Latent.import_local_latent relies on local_latent.export dict.

# Let's populate generic labels for discovered clusters
if hasattr(target, 'cluster'):
    unique_clusters = np.unique(target.cluster)
    for cid in unique_clusters:
        if cid == -1: continue
        target.label_cluster(cid, f'Cluster_{cid}', target.palette(cid))

    X.import_local_latent(target)
    print("Clusters imported.")

In [ ]:
# Plot Ethogram (Syllables)
plt.figure(figsize=(15, 3))
X.plot_syllables()
plt.title('Behavior Ethogram')
plt.xlabel('Frame')
plt.tight_layout()
plt.show()